# initiating 

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import PercentFormatter
import networkx as nx
import pandas as pd
from os import listdir
import xlrd
import warnings
from ipynb.fs.full.Reading_CloudNetworks import SNetwork_DictList , Networks_DictList



In [ ]:
def plot(results, title,victim,names,type):
    if type =="NCC": TData=2 ; titl='NCC'
    elif type =="LCC": TData=3; titl='$|N_{LCC}| / |N|$ '
    elif type =="diam": TData=4; titl='Diameter '
    else:
       warnings.warn("Warning...uncorrect type /n type parameter only takes NCC,LCC or diam /n value this function only plot: NCC,LCC and diameter, ") 
    fig = plt.figure(dpi=600)
    fig, ax = plt.subplots()
    markers = ['+', 'x', 'o', 's', 'd', 'D', '*',] # Add your desired markers here
    for d in range(len(results)):
        data = results[d]
        x = [(item[1])*10 for item in data]
        y = [item[TData] for item in data]
        marker_idx = d % len(markers) # Choose marker based on index of the result
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(100,0)) 
        ax.plot(x, y, marker=markers[marker_idx], linewidth=0.5, markersize=6,label=names[d])

    ax.set_xlabel(f'{victim} Removed')
    ax.set_ylabel(f'{titl}')
    ax.set_title(f'{title}  ')
    plt.legend()
    plt.legend(bbox_to_anchor=(0.5, 1.35), loc='upper center',ncol=3)

In [ ]:
#used in random attack code 
def batch_list_per(lst,per):
    NSize=len(lst)
    # Divide a list into batches based of the percentage specified in the parameter
    batch_size = round(NSize*per) # each patch size is x% of the netweok size 
    num_batches = (len(lst) + batch_size - 1) // batch_size
    batches = [lst[i*batch_size:(i+1)*batch_size] for i in range(num_batches)]
    return batches


#used in targeted attack code 
def cal_batchSize_per(lst,per):
    NSize=len(lst)
    batch_size = round(NSize*per)
    return batch_size

def simple_batching(lst,batch_size): 
    num_batches = (len(lst) + batch_size - 1) // batch_size
    batches = [lst[i*batch_size:(i+1)*batch_size] for i in range(num_batches)]
    return batches

In [ ]:
def properties(G):
#return properties of graph G if the gaint compponet did not disapear or only contined 1 node 

    Connected_components = list(nx.connected_components(G))
    numConnected_components= len(Connected_components)
    if numConnected_components > 0:
        subG = G.subgraph(max(Connected_components, key=len))   
        if len(subG.nodes())>1: 
            # for Avrage Shortes Path and diameter wil we will only consider the gaint component  
            ASP = nx.average_shortest_path_length(subG)
            degrees = nx.degree_centrality(G)
            # de-normalizing degree values to calculate the avarage 
            dList= degrees.values()
            maxpoosibleD=G.number_of_nodes()-1
            deNormalize= [x * maxpoosibleD for x in dList]
            avg_degree= round(sum(deNormalize)/G.number_of_nodes(),1)
            diam = nx.diameter(subG)
            ACC = nx.average_clustering(G)
            numEdges= len(G.edges()) #does not work with edges attacks only for networks proporties 

            return [numConnected_components,len(subG) ,ASP, diam,ACC, avg_degree, numEdges]
        
        else:
            return None, None,None,None, None,None,None
        
    else:
        # Handle the case when there are no connected components (keep as our method also use LLC as an andicator to when shall we stop the attack, see [our paper])
        return None, None, None,None, None,None,None

In [ ]:
# def propertiesDeg(G):
# #return properties of graph G if the gaint compponet did not disapear or only contined 1 node 

#     Connected_components = list(nx.connected_components(G))
#     numConnected_components= len(Connected_components)
#     if numConnected_components > 0:
#         subG = G.subgraph(max(Connected_components, key=len))   
#         if len(subG.nodes())>1: 
            
#             degrees = nx.degree_centrality(G)
#             # de-normalizing degree values
#             dList= degrees.values()
#             maxpoosibleD=G.number_of_nodes()-1
#             deNormalize= [x * maxpoosibleD for x in dList]
#             avg_degree= round(sum(deNormalize)/G.number_of_nodes(),1)
            

#             return [numConnected_components,len(subG), avg_degree]
        
#         else:
#             return None, None,None
        
#     else:
#         # Handle the case when there are no connected components (keep as our method also use LLC as an andicator to when shall we stop the attack, see [our paper])
#         return None, None, None

# Random Attacks

## Nodes Random Attacks 

In [ ]:
networks = Networks_DictList()
names=[]
for network in networks:
        names.append(network['name'])

def randomN_AttckSim():
    results = []  
    for network in networks:
        G = network['graph'].copy()
        V = list(G.nodes)
        random.shuffle(V)
        batches = batch_list_per(V,0.1)
        result = []
        for b in range(len(batches)):
            G.remove_nodes_from(batches[b])
            if b<5:
                umConnected_components,largest_componentSize ,ASP, diam,ACC, avg_degree,numEdges= properties(G)
                result.append([network['name'],b, umConnected_components ,largest_componentSize/len(G) ,ASP, diam,ACC, avg_degree])
        results.append(result)
    return results



In [ ]:
VRA_Table = pd.DataFrame()
VRA_name= [];VRA_numofBatch=[];VRA_umConnected_components=[];VRA_LCC=[];VRA_diam= [];VRA_ASP=[]; VRA_ACC=[]; VRA_avg_degree=[]
rndV_results = randomN_AttckSim()


for result in  rndV_results:
    for batches in result:
        VRA_name.append(batches[0])
        VRA_numofBatch.append(f"{((batches[1]+1)*10)} %")
        VRA_umConnected_components.append(batches[2])
        VRA_LCC.append(batches[3])
        VRA_diam.append(batches[5])
        VRA_ASP.append(batches[4])
        VRA_ACC.append(batches[6])
        VRA_avg_degree.append(batches[7])

VRA_Table["Networks"]=VRA_name; VRA_Table["Percentage of nodes removed"]= VRA_numofBatch
VRA_Table["Number of Connected Components"]= VRA_umConnected_components
VRA_Table["largest_component/N"]=VRA_LCC ;VRA_Table["Diameter of the LCC"]=VRA_diam
VRA_Table["avrage shortes path of the LCC"] = VRA_ASP; VRA_Table["Average Clustering Coefficient"] =VRA_ACC
VRA_Table["avrage Degress"] = VRA_avg_degree 

In [ ]:
VRA_Table

In [ ]:
VRA_Table.to_csv('Random_nodesAttcakDeg.csv')


In [ ]:
Plot_types=['NCC',"LCC","diam"]
for type in Plot_types:
    plot(rndV_results, 'Random Attack', 'Nodes',names,type)

# for NCC divid by the netwrok size to get nornilized resluts
# for diamater ? does it make sense ?     

## Edges Random Attacks 

In [ ]:
networks = Networks_DictList()
names=[]
for network in networks:
        names.append(network['name'])

def randomE_AttckSim():
    results = []
    for network in networks:
        G = network['graph'].copy()
        E = list(G.edges)
        random.shuffle(E)
        batches = batch_list_per(E,0.1)
        result = []
        for b in range(len(batches)):
            G.remove_edges_from(batches[b])            
            if b<5:
                umConnected_components,largest_componentSize ,ASP, diam,ACC, avg_degree,numEdges= properties(G)
                result.append([network['name'],b, umConnected_components ,largest_componentSize/len(G) ,ASP, diam,ACC, avg_degree])

        results.append(result)
    return results

In [ ]:
ERA_Table = pd.DataFrame()
ERA_name= [];ERA_numofBatch=[];ERA_umConnected_components=[];ERA_LCC=[];ERA_diam= [];ERA_ASP=[]; ERA_ACC=[]; ERA_avg_degree=[]
rndE_results = randomE_AttckSim()


for result in  rndE_results:
    for batches in result:
        ERA_name.append(batches[0])
        ERA_numofBatch.append(f"{((batches[1]+1)*10)} %")
        ERA_umConnected_components.append(batches[2])
        ERA_LCC.append(batches[3])
        ERA_diam.append(batches[5])
        ERA_ASP.append(batches[4])
        ERA_ACC.append(batches[6])
        ERA_avg_degree.append(batches[7])

ERA_Table["Networks"]=ERA_name; ERA_Table["Percentage of Edges removed"]= ERA_numofBatch
ERA_Table["Number of Connected Components"]= ERA_umConnected_components
ERA_Table["largest_component/N"]=ERA_LCC ;ERA_Table["Diameter of the LCC"]=ERA_diam
ERA_Table["avrage shortes path of the LCC"] = ERA_ASP; ERA_Table["Average Clustering Coefficient"] =ERA_ACC
ERA_Table["avrage Degress"] = ERA_avg_degree 

ERA_Table

In [ ]:
ERA_Table.to_csv('Random_EdgesAttcak.csv')

In [ ]:
Plot_types=['NCC',"LCC","diam"]
for type in Plot_types:
    plot(rndE_results, 'Random Attack', 'Edges',names,type)

# Targeted Attacks

In [ ]:
centr = ['Degree','Betweenness']

#return an order list of only the only one CI specified by the paramenter low to hight for recusive method 
def CI_Node_orderd_list2(G,centrality ):
    CI_Dict =centrality(G)
    CI_sortedDict = {k: v for k, v in sorted(CI_Dict.items(), key=lambda item: item[1])}
    CI_sortedList= list(CI_sortedDict.keys())
    return CI_sortedList  

#  return an order edge list of only the only one CI specified by the paramenter low to hight 
def weighted_edges(G, centrality): #  from [8] for degree only + other if for betweeness
    '''return a weighted edges equations from paper 11'''
    W = []

    if centrality == nx.betweenness_centrality:
        CI_Dict = nx.edge_betweenness_centrality(G)
        CI_sortedDict = {k: v for k, v in sorted(CI_Dict.items(), key=lambda item: item[1])}
        for u,v in G.edges():
            W.append([u, v, CI_sortedDict[u,v]] )
    else:
        CI_Dict = centrality(G)
        for u,v in G.edges():
            W.append([u, v, CI_Dict[u]*CI_Dict[v]])
    return sorted(W, key=lambda x: x[2])


In [ ]:
def recalculateRecursionGen(network,G,batches,batches_size,centrality,i,results, victim):

    print("network before removal of 10%",network['name'])
    if victim == "Node Attack":
        R = batches[i] # Nodes to be removed...
        G.remove_nodes_from(R)
    else:
        R = [(u,v) for u,v,_ in batches[i]] # edges to be removed...
        G.remove_edges_from(R)

    if len(results)<5:
        umConnected_components,largest_componentSize ,ASP, diam,ACC, avg_degree,numEdges= properties(G)
        if largest_componentSize:
            print("network After removal of 10%",network['name'])
            results.append([network['name'],len(results)+1, umConnected_components ,largest_componentSize/len(G),ASP, diam,ACC, avg_degree])       
    
    i-=1

    if victim == "Node Attack":
        Ranked_lists = CI_Node_orderd_list2(G,centrality)
    else:    
        Ranked_lists = weighted_edges(G,centrality) 
    
    neWbatches = simple_batching(Ranked_lists,batches_size) # batchin the nodes into an BN of batches 
    if  i<=0  : return results
    else: return recalculateRecursionGen(network,G, neWbatches,batches_size,centrality, i ,results,victim)

## Nodes Targeted Attacks 

In [ ]:
# adding recalculating after each batch removal
 
networks = Networks_DictList()
names=[]
for network in networks:
        names.append(network['name'])
        
def simulationREC(centrality,per):
    results = []  
    for network in networks:
        G = network['graph'].copy()
        Ranked_lists = CI_Node_orderd_list2(G,centrality) 
        batches_size= cal_batchSize_per(Ranked_lists,per)   
        batches = simple_batching(Ranked_lists,batches_size)
        result =recalculateRecursionGen(network,G,batches,batches_size,centrality,len(batches)-1,[],"Node Attack")
        results.append(result)

    return results

In [ ]:
centralities = [nx.degree_centrality, nx.betweenness_centrality]
for i in range(len(centralities)):
    
    TA_results = simulationREC(centralities[i],0.1)

    Plot_types=['NCC',"LCC","diam"]
    for type in Plot_types:
        plot(TA_results, f'targated {centr[i]} Attack', 'Nodes',names,type)
        
    
    VTA_Table = pd.DataFrame()
    VTA_name= [];VTA_numofBatch=[];VTA_umConnected_components=[];VTA_LCC=[];VTA_diam= [];VTA_ASP=[]; VTA_ACC=[]; VTA_avg_degree=[]
    for result in  TA_results:
        for batches in result:
            VTA_name.append(batches[0])
            VTA_numofBatch.append(f"{((batches[1])*10)} %")
            VTA_umConnected_components.append(batches[2])
            VTA_LCC.append(batches[3])
            VTA_diam.append(batches[5])
            VTA_ASP.append(batches[4])
            VTA_ACC.append(batches[6])
            VTA_avg_degree.append(batches[7])

    VTA_Table["Networks"]=VTA_name; VTA_Table["Percentage of nodes removed"]= VTA_numofBatch
    VTA_Table["Number of Connected Components"]= VTA_umConnected_components
    VTA_Table["largest_component/N"]=VTA_LCC ;VTA_Table["Diameter"]=VTA_diam
    VTA_Table["avrage shortes path"] = VTA_ASP; VTA_Table["Average Clustering Coefficient"] =VTA_ACC
    VTA_Table["avrage Degress"] = VTA_avg_degree
    VTA_Table
    VTA_Table.to_csv(f'Targated_{centr[i]}_nodesAttcak.csv')
VTA_Table

## Edges Targeted Attacks 

In [ ]:
networks = Networks_DictList()
names=[]
for network in networks:
    names.append(network['name'])
        
def E_simulationREC(centrality,per):
    results = []  
    for network in networks:
        G = network['graph'].copy()
        Ranked_lists = weighted_edges(G,centrality) 
        batches_size= cal_batchSize_per(Ranked_lists,per)   
        batches = simple_batching(Ranked_lists,batches_size)
        result =recalculateRecursionGen(network,G,batches,batches_size,centrality,len(batches)-1,[],"Edge attck")
        results.append(result)

    return results

In [ ]:
centralities = [nx.degree_centrality, nx.betweenness_centrality]
Plot_types=['NCC',"LCC","diam"]
for i in range(len(centralities)):
    ETA_results = E_simulationREC(centralities[i],0.1)   

    for type in Plot_types:
        plot(ETA_results, f'targated {centr[i]} Attack', 'Edges',names,type)
    
    ETA_Table = pd.DataFrame()
    ETA_name= [];ETA_numofBatch=[];ETA_umConnected_components=[];ETA_LCC=[];ETA_diam= [];ETA_ASP=[]; ETA_ACC=[]; ETA_avg_degree=[]
    for result in  ETA_results:
        for batches in result:
            ETA_name.append(batches[0])
            ETA_numofBatch.append(f"{((batches[1])*10)} %")
            ETA_umConnected_components.append(batches[2])
            ETA_LCC.append(batches[3])
            ETA_diam.append(batches[5])
            ETA_ASP.append(batches[4])
            ETA_ACC.append(batches[6])
            ETA_avg_degree.append(batches[7])

    ETA_Table["Networks"]=ETA_name; ETA_Table["Percentage of nodes removed"]= ETA_numofBatch
    ETA_Table["Number of Connected Components"]= ETA_umConnected_components
    ETA_Table["largest_component/N"]=ETA_LCC ;ETA_Table["Diameter"]=ETA_diam
    ETA_Table["avg shortes path"] = ETA_ASP; ETA_Table["Average Clustering Coefficient"] =ETA_ACC
    ETA_Table["avg Degress"] = ETA_avg_degree
    ETA_Table.to_csv(f'Targated {centr[i]}_EdgeAttcak.csv')
ETA_Table